# DataRetail360: flujo completo Bronze, Silver y Gold

Ejecutamos las mismas funciones del proceso completo, una etapa a la vez.
Bronze conserva los CSV, Silver genera siete entidades con claves de negocio
y Gold construye el modelo dimensional después de releer Parquet.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src" / "proceso.py").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.proceso import SesionETL
from src.carga.gold.conexion import crear_motor

sesion = SesionETL(RAIZ, modo="manual")
print("Ejecución:", sesion.id_ejecucion)

Ejecución: ETL-c8672d5d0401


## 1. Extraer Bronze

Verificamos columnas, filas y SHA-256 de los siete CSV. Todavía no se
normaliza ni se descarta ninguna fila.

In [2]:
fuentes = sesion.extraer()
display(pd.DataFrame([
    {"fuente": nombre, "filas": len(datos), "columnas": len(datos.columns)}
    for nombre, datos in fuentes.items()
]))
display(fuentes["ventas"].head(3))

,fuente,filas,columnas
0,sucursales,22,8
1,productos,1000,7
2,clientes,10100,5
3,cambios_cliente,2000,4
4,cambios_producto,50,6
5,ventas,125854,13
6,devoluciones,4618,8


,sistema_origen,id_comprobante,numero_linea,fecha_venta,id_cliente,codigo_producto,codigo_sucursal,cantidad,precio_unitario,porcentaje_descuento,importe_bruto,importe_tras_descuento,estado
0,POS,V0000001,1,2025-10-26,C003451,P00619,T002,1,172700.0,0,172700,172700,VALIDA
1,ECOM,V0000002,1,2025-07-17,C004644,P00815,APP01,3,795200.0,0,2385600,2385600,VALIDA
2,ECOM,V0000002,2,2025-07-17,C004644,P00140,APP01,1,526700.0,0,526700,526700,VALIDA


## 2. Limpiar y conciliar Silver

Normalizamos identificadores y tipos, retiramos copias exactas y enviamos
errores a auditoría. Las ventas y devoluciones conservan claves de negocio;
aún no existen claves sustitutas ni tablas dimensionales.

In [3]:
tablas_silver = sesion.limpiar_silver()
auditoria = sesion.conciliar_silver()
display(pd.DataFrame([
    {"entidad_silver": nombre, "filas": len(datos)}
    for nombre, datos in tablas_silver.items()
]))
display(pd.DataFrame(auditoria["conciliacion"]))
display(pd.DataFrame(auditoria["rechazos"])[["fuente", "motivo"]].value_counts().reset_index(name="filas"))

,entidad_silver,filas
0,clientes_limpios,10000
1,cambios_cliente_limpios,2000
2,productos_limpios,1000
3,cambios_producto_limpios,50
4,sucursales_limpias,22
5,ventas_limpias,125217
6,devoluciones_limpias,4612


,fuente,filas_leidas,filas_aceptadas,filas_rechazadas,filas_duplicadas,diferencia
0,clientes,10100,10000,0,100,0
1,cambios_cliente,2000,2000,0,0,0
2,productos,1000,1000,0,0,0
3,cambios_producto,50,50,0,0,0
4,sucursales,22,22,0,0,0
5,ventas,125854,125217,11,626,0
6,devoluciones,4618,4612,5,1,0


,fuente,motivo,filas
0,ventas,conflicto_clave_negocio,2
1,ventas,campo_obligatorio_vacio,1
2,ventas,producto_desconocido,1
3,ventas,medida_no_positiva,1
4,ventas,fecha_venta_invalida,1
5,ventas,descuento_invalido,1
6,ventas,importe_incoherente,1
7,ventas,cliente_desconocido,1
8,ventas,estado_no_admitido,1
9,ventas,valor_numerico_invalido,1


## 3. Guardar y volver a leer Silver

Escribimos siete Parquet Snappy, auditoría y manifiesto. Después descartamos
la copia previa y comprobamos que Gold usa la lectura física.

In [4]:
manifiesto = sesion.guardar_silver()
display(pd.DataFrame(manifiesto["entidades"])[["entidad", "archivo", "filas", "sha256"]])

tablas_silver = sesion.leer_silver()
claves_gold = {"clave_fecha", "clave_producto", "clave_cliente",
               "clave_sucursal_canal", "clave_venta", "clave_devolucion"}
assert all(not claves_gold.intersection(datos.columns) for datos in tablas_silver.values())
print("Silver releída desde disco y sin claves sustitutas Gold.")

,entidad,archivo,filas,sha256
0,clientes_limpios,clientes_limpios.parquet,10000,7f0397369205d800ae492fc9ffcd2caa8f4a72ec45ee88...
1,cambios_cliente_limpios,cambios_cliente_limpios.parquet,2000,8e0b7783f250020fd7520630f1864670cda0b0cf283095...
2,productos_limpios,productos_limpios.parquet,1000,ec873a1ca67ddede7bae1999351bdcb9b4d770102107e1...
3,cambios_producto_limpios,cambios_producto_limpios.parquet,50,041dc4f0636e871c4f2a380795ef016b806b89d6c997b1...
4,sucursales_limpias,sucursales_limpias.parquet,22,14ad1c098b360103f223f6d722e49075b9e6be7c8c0c66...
5,ventas_limpias,ventas_limpias.parquet,125217,7f22b2ce30ad336e51023028e87bb4d957563a8f27281c...
6,devoluciones_limpias,devoluciones_limpias.parquet,4612,abdc9b4249877d17d483c974b173fe020d007659e22557...


Silver releída desde disco y sin claves sustitutas Gold.


## 4. Construir Gold

Aplicamos SCD 1 y 3 a productos, SCD 2 a clientes, construimos calendario
y resolvemos las claves de los dos hechos.

In [5]:
dimensiones = sesion.construir_dimensiones_gold()
hechos = sesion.construir_hechos_gold()
tablas_gold = sesion.conciliar_gold()
display(pd.DataFrame([
    {"tabla_gold": nombre, "filas": len(datos)}
    for nombre, datos in tablas_gold.items()
]))
display(dimensiones["dim_cliente"].head(3))
display(hechos["hecho_ventas"].head(3))

,tabla_gold,filas
0,dim_fecha,731
1,dim_producto,1000
2,dim_cliente,12000
3,dim_sucursal_canal,22
4,hecho_ventas,125217
5,hecho_devoluciones,4612


,clave_cliente,id_cliente,correo_normalizado,region,segmento,vigente_desde,vigente_hasta,es_actual
0,1,C000001,cliente1@example.invalid,Metropolitana,Ocasional,2024-01-01,2024-06-29,0
1,2,C000001,cliente1@example.invalid,Valparaiso,Frecuente,2024-06-29,2025-03-26,0
2,3,C000001,cliente1@example.invalid,Biobio,Preferente,2025-03-26,9999-12-31,1


,clave_venta,clave_negocio,sistema_origen,id_comprobante,numero_linea,clave_fecha,clave_producto,clave_cliente,clave_sucursal_canal,cantidad,precio_unitario,porcentaje_descuento,importe_bruto,importe_descuento,importe_tras_descuento,id_ejecucion
0,1,ECOM|V0000002|1,ECOM,V0000002,1,20250717,815,6644,1,3,795200,0,2385600,0,2385600,ETL-c8672d5d0401
1,2,ECOM|V0000002|2,ECOM,V0000002,2,20250717,140,6644,1,1,526700,0,526700,0,526700,ETL-c8672d5d0401
2,3,ECOM|V0000007|1,ECOM,V0000007,1,20250803,534,9891,22,2,783200,0,1566400,0,1566400,ETL-c8672d5d0401


## 5. Cargar, validar y publicar SQLite

Creamos una base temporal, cargamos dimensiones antes que hechos, ejecutamos
controles SQL y reemplazamos la base final únicamente si todos terminan en cero.

In [6]:
sesion.conectar()
sesion.crear_modelo()
sesion.cargar_dimensiones()
sesion.cargar_hechos()
controles = sesion.validar()
display(pd.DataFrame([
    {"control": nombre, "infracciones": valor}
    for nombre, valor in controles.items()
]))
display(pd.Series(sesion.resumen["metricas"], name="valor").to_frame())

,control,infracciones
0,versiones_cliente_solapadas,0
1,versiones_actuales_invalidas,0
2,version_cliente_incorrecta,0
3,exceso_devoluciones,0
4,devolucion_anterior_venta,0
5,dimensiones_devolucion_incoherentes,0
6,claves_venta_duplicadas,0
7,importe_venta_invalido,0
8,referencias_huerfanas,0
9,descuadre_conciliacion,0


,valor
ventas_brutas,9.469950e+10
descuentos,6.758099e+09
ventas_tras_descuento,8.794140e+10
devoluciones,1.633664e+09
ventas_netas,8.630774e+10
comprobantes,4.999900e+04
ticket_promedio_antes_devoluciones,1.758863e+06


In [7]:
resumen = sesion.publicar()
assert resumen["estado"] == "CORRECTA"
print("Base publicada:", sesion.ruta_base)
print("Ventas Silver/Gold:", resumen["conciliacion_gold"]["ventas_silver"],
      resumen["conciliacion_gold"]["ventas_gold"])
print("Devoluciones Silver/Gold:", resumen["conciliacion_gold"]["devoluciones_silver"],
      resumen["conciliacion_gold"]["devoluciones_gold"])

Base publicada: C:\Users\diego\OneDrive\Escritorio\Peter\Datos_oficiales\DataRetail360\resultados\demostracion.sqlite
Ventas Silver/Gold: 125217 125217
Devoluciones Silver/Gold: 4612 4612


## 6. Consultar el resultado

Las vistas evitan multiplicar ventas cuando existen varias devoluciones y
permiten explicar actividad mensual, productos y sucursales.

In [8]:
motor = crear_motor(sesion.ruta_base)
try:
    with motor.connect() as conexion:
        display(pd.read_sql_query(
            "SELECT anio, mes, canal, ventas_netas "
            "FROM vista_actividad_mensual ORDER BY anio, mes, canal LIMIT 12",
            conexion,
        ))
        display(pd.read_sql_query(
            "SELECT codigo_producto, nombre_producto, ventas_netas "
            "FROM vista_ventas_por_producto ORDER BY ventas_netas DESC LIMIT 10",
            conexion,
        ))
finally:
    motor.dispose()

,anio,mes,canal,ventas_netas
0,2024,1,E-commerce,1016975470
1,2024,1,Tienda,2490508290
2,2024,2,E-commerce,909311125
3,2024,2,Tienda,2163761585
4,2024,3,E-commerce,970413510
5,2024,3,Tienda,2317590570
6,2024,4,E-commerce,954672170
7,2024,4,Tienda,2282666225
8,2024,5,E-commerce,848675850
9,2024,5,Tienda,2350209225


,codigo_producto,nombre_producto,ventas_netas
0,P00815,Cocina modelo 815,228898320
1,P00534,Cables modelo 534,218003720
2,P00200,Electrodomesticos modelo 200,215336310
3,P00672,Fundas modelo 672,213852000
4,P00692,Muebles modelo 692,212317105
5,P00159,Fundas modelo 159,212134050
6,P00016,Nombre corregido 16,204457680
7,P00440,Muebles modelo 440,202797650
8,P00303,Fundas modelo 303,200825100
9,P00845,Muebles modelo 845,199327960
